# Tracing with mlflow

## Environment Set-up

Create a virtual environment:

```
python -m venv .venv
```

Activate the virtual environment:

```
source .venv/bin/activate
```

Install the dependencies

```
pip install -r requirements.txt
```

Start mlflow server:

```
mlflow server --host 0.0.0.0 --port 2000
```

In [2]:
# Connect to mlflow server and set experiment

import mlflow

mlflow.set_tracking_uri("http://localhost:2000")
mlflow.set_experiment("Tracing Quickstart")

2025/10/08 14:02:57 INFO mlflow.tracking.fluent: Experiment with name 'Tracing Quickstart' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/641425381063492368', creation_time=1759892577179, experiment_id='641425381063492368', last_update_time=1759892577179, lifecycle_stage='active', name='Tracing Quickstart', tags={}>

Go to http://localhost:2000 to access the mlflow UI

In [3]:
# Load environment variables

import dotenv

dotenv.load_dotenv()

True

## Automatic tracing

Easy one-line tracing

In [15]:
# Tracing with OpenAI

import mlflow
from openai import OpenAI

mlflow.openai.autolog() # One-liner tracing

client = OpenAI()

client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

2025/10/08 15:09:12 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


ChatCompletion(id='chatcmpl-COFmCVffp9Z8Vdr7V4sRPA3phapbS', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1759896552, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_7c233bf9d1', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

Trace(trace_id=tr-902736107d4aed31369cbb3ef2524799)

## Manual tracing

If you want to trace more complex workflows, mlflow exposes the very useful `@mlflow.trace` decorator which you can use to trace any function.

In [5]:
@mlflow.trace(name="add",)
def add(x, y):
    return x + y

In [6]:
add(1, 2)

2025/10/08 14:03:01 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


3

Trace(trace_id=tr-54974741c9359e32c287f7f6f82954dc)

## Agentic workflow + Tracing

Let's look at a more complicated workflow which will involve function calling. We will trace this tool.

In [16]:
import requests
from mlflow.entities import SpanType

# Add decorator to trace the tool
@mlflow.trace(span_type=SpanType.TOOL)
def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]["temperature_2m"]

In [8]:
# Define OpenAI tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [ ]:
# Create a function to run the agentic workflow

import json
from mlflow.entities import SpanType

@mlflow.trace(span_type=SpanType.AGENT)
def run_weather_agent(question: str):
    messages = [
        {"role": "user", "content": question}]

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        tools=tools,
    )
    ai_msg = response.choices[0].message
    messages.append(ai_msg)

    if tool_calls := ai_msg.tool_calls:
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            if function_name == "get_weather":
                kwargs = json.loads(tool_call.function.arguments)
                tool_result = get_weather(**kwargs)
            else:
                raise RuntimeError("An invalid tool is returned from the LLM")
            
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result)
                }
            )

        response = client.chat.completions.create(model="o4-mini", messages=messages)

    return response.choices[0].message.content
            

In [17]:
# Run the agentic workflow

question = "What is the weather in Melbourne?"
run_weather_agent(question)

2025/10/08 15:14:07 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 15:14:08 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 15:14:12 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 15:14:12 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


'The current temperature in Melbourne is 20 °C.'

Trace(trace_id=tr-09555e8d8256d2401382a24674fa0e86)

## SpanType

[SpanType API Reference](https://mlflow.org/docs/3.2.0/api_reference/python_api/mlflow.entities.html?highlight=spantype#mlflow.entities.SpanType)

In [11]:
import anthropic

ant = anthropic.Anthropic()

In [14]:
# Combining manual tracing and automatic tracing

mlflow.anthropic.autolog()

@mlflow.trace(span_type=SpanType.CHAIN)
def run_chain(query: str):
    messages = build_messages(query)

    # Anthropic auto tracing will take care of this call
    response = ant.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        messages=messages,
    )

    return parse_response(response)

@mlflow.trace
def build_messages(query: str):
    return [
        {"role": "user", "content": query},
    ]

@mlflow.trace
def parse_response(response):
    return response.content[0].text

run_chain("What is the study of fluid dynamics?")

2025/10/08 14:08:26 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 14:08:31 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 14:08:31 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/08 14:08:31 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


"Fluid dynamics is a branch of physics that deals with the study of fluids (liquids and gases) in motion and the interactions between fluids and solid surfaces. It focuses on understanding how fluids behave when they are subjected to various forces and conditions. Key aspects of fluid dynamics include:\n\n1. Flow characteristics\n- Velocity\n- Pressure\n- Density\n- Temperature\n- Viscosity\n\n2. Types of fluid flow\n- Laminar flow\n- Turbulent flow\n- Compressible and incompressible flow\n- Steady and unsteady flow\n\n3. Fundamental principles\n- Conservation of mass\n- Conservation of momentum\n- Conservation of energy\n- Bernoulli's equation\n\n4. Applications\n- Aerodynamics\n- Hydrodynamics\n- Meteorology\n- Oceanography\n- Automotive and aerospace engineering\n- Blood flow in medical research\n- Weather prediction\n- Design of pipes, pumps, and turbines\n\n5. Mathematical and computational modeling\n- Navier-Stokes equations\n- Computational fluid dynamics (CFD)\n\nFluid dynamics

Trace(trace_id=tr-15fac4ccd705b1fa3a8e0861c12dc509)

# Complex Agent

Let's build a complex agent which does tool calls, uses multiple providers, and has access to a vector database - adding tracing to everything.

In [ ]:
# Set-up vector database

import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.create_collection("documents")

collection.add(
    ids=["id1", "id2", "id3"],
    documents=[
        "Travel time: One-way flight from Vandul to Farish is 1 hour 30 minutes. One-way flight Equoza to Opinstian is 3 hours 45 minutes. One-way flight from Maddox to Lenivin is 8 hours.",
        "Essential packing: "
    ]
)